# VATSA A-Module — Experiment 001: Frozen Wav2Vec2 Baseline

**Phase:** 2 — A-Module (Audio Encoder)
**Date:** May 2026
**Objective:** Establish baseline using Wav2Vec2 as pure feature extractor with frozen backbone. Train projection head + classifier on Speech Commands dataset.

**Parallel to V-Module:** Experiment 001 (Frozen EfficientNet Baseline)

## 1. Setup & Imports

In [13]:
import torch
import torch.nn as nn
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import DataLoader, Dataset
from transformers import Wav2Vec2Model, Wav2Vec2Processor
import transformers
import os
import numpy as np
from pathlib import Path
import json
from datetime import datetime

# Force W&B to stay offline (same as V-Module)
os.environ["WANDB_MODE"] = "offline"
import wandb

# Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# VATSA shared latent dimension
VATSA_EMBEDDING_DIM = 512

print(f"transformers: {transformers.__version__}")
print(f"torchaudio: {torchaudio.__version__}")

Using device: cuda
transformers: 5.7.0
torchaudio: 2.6.0+cu124


## 2. Dataset: Speech Commands v0.02

Lightweight benchmark: 35 word classes, ~105k utterances, 1-second clips at 16kHz.

**Parallel to V-Module:** CIFAR-10 (10 classes, 60k images)

In [7]:
# Download Speech Commands dataset
# This will download ~2GB on first run
dataset_path = "C:\\Users\\vinay\\OneDrive\\Desktop\\VATSA\\VATSA\\notebooks\\data"

train_dataset = torchaudio.datasets.SPEECHCOMMANDS(
    root=dataset_path,
    url="speech_commands_v0.02",
    subset="training",
    download=True
)

val_dataset = torchaudio.datasets.SPEECHCOMMANDS(
    root=dataset_path,
    url="speech_commands_v0.02",
    subset="validation",
    download=True
)

test_dataset = torchaudio.datasets.SPEECHCOMMANDS(
    root=dataset_path,
    url="speech_commands_v0.02",
    subset="testing",
    download=True
)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

Train: 84843 | Val: 9981 | Test: 11005


In [8]:
# Build label mapping (same pattern as V-Module CIFAR-10 classes)
all_labels = sorted(list(set([item[2] for item in train_dataset])))
label_to_idx = {label: idx for idx, label in enumerate(all_labels)}
num_classes = len(all_labels)

print(f"Number of classes: {num_classes}")
print(f"Classes: {all_labels}")

Number of classes: 35
Classes: ['backward', 'bed', 'bird', 'cat', 'dog', 'down', 'eight', 'five', 'follow', 'forward', 'four', 'go', 'happy', 'house', 'learn', 'left', 'marvin', 'nine', 'no', 'off', 'on', 'one', 'right', 'seven', 'sheila', 'six', 'stop', 'three', 'tree', 'two', 'up', 'visual', 'wow', 'yes', 'zero']


## 3. Audio Preprocessing

Wav2Vec2 expects raw waveform at 16kHz. We pad/truncate to consistent length.

**Parallel to V-Module:** Resize(256) → CenterCrop(224) → Normalize

In [9]:
class AudioPreprocessor:
    """
    Preprocess raw audio for Wav2Vec2:
    - Resample to 16kHz (if needed)
    - Pad/truncate to target_length samples
    - Normalize amplitude
    """
    def __init__(self, target_length=16000, sample_rate=16000):
        self.target_length = target_length
        self.sample_rate = sample_rate
        self.resampler = T.Resample(orig_freq=44100, new_freq=sample_rate)
    
    def __call__(self, waveform, orig_sample_rate):
        # Resample if needed
        if orig_sample_rate != self.sample_rate:
            if orig_sample_rate == 44100:
                waveform = self.resampler(waveform)
            else:
                resample = T.Resample(orig_freq=orig_sample_rate, new_freq=self.sample_rate)
                waveform = resample(waveform)
        
        # Mono channel
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        
        # Pad or truncate
        if waveform.shape[1] < self.target_length:
            padding = self.target_length - waveform.shape[1]
            waveform = torch.nn.functional.pad(waveform, (0, padding))
        else:
            waveform = waveform[:, :self.target_length]
        
        # Normalize
        waveform = (waveform - waveform.mean()) / (waveform.std() + 1e-8)
        
        return waveform.squeeze(0)  # (target_length,)

preprocessor = AudioPreprocessor(target_length=16000)

In [10]:
class SpeechCommandsDataset(Dataset):
    """
    Wrapper for Speech Commands with preprocessing.
    """
    def __init__(self, base_dataset, label_to_idx, preprocessor):
        self.base_dataset = base_dataset
        self.label_to_idx = label_to_idx
        self.preprocessor = preprocessor
    
    def __len__(self):
        return len(self.base_dataset)
    
    def __getitem__(self, idx):
        waveform, sample_rate, label, *_ = self.base_dataset[idx]
        
        # Preprocess
        waveform = self.preprocessor(waveform, sample_rate)
        
        # Label
        label_idx = self.label_to_idx[label]
        
        return waveform, label_idx

# Wrap datasets
train_data = SpeechCommandsDataset(train_dataset, label_to_idx, preprocessor)
val_data = SpeechCommandsDataset(val_dataset, label_to_idx, preprocessor)
test_data = SpeechCommandsDataset(test_dataset, label_to_idx, preprocessor)

# DataLoaders (num_workers=2 for Windows, same as V-Module)
batch_size = 256  # Adjust based on GPU memory
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2, persistent_workers=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False, num_workers=2, persistent_workers=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=2, persistent_workers=True)

print(f"Batch size: {batch_size}")
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")

Batch size: 256
Train batches: 332 | Val batches: 39 | Test batches: 43


## 4. VATSA Audio Encoder

**Parallel to V-Module:** `VATSA_VisualEncoder` — same structure, different backbone.

- **Backbone:** Wav2Vec2-base (pretrained on 960h Librispeech)
- **Projection:** 768 → 512 (shared latent space)
- **Norm:** LayerNorm(512)
- **Classifier:** Dropout(0.3) + Linear(512, 35)

In [14]:
class VATSA_AudioEncoder(nn.Module):
    """
    VATSA Audio Module
    Projects audio/speech into 512-dim shared latent space.
    Future-ready for multimodal fusion.
    """
    def __init__(
        self,
        embedding_dim: int = 512,
        backbone_name: str = "wav2vec2_base",
        pretrained: bool = True,
        freeze_backbone: bool = True,
        num_classes: int = 35
    ):
        super().__init__()
        
        # Load pretrained backbone
        if backbone_name == "wav2vec2_base":
            self.backbone = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base-960h")
            self.backbone_feature_dim = 768
        else:
            raise ValueError(f"Unsupported backbone: {backbone_name}")
        
        # Remove Wav2Vec2's built-in feature extractor projection if needed
        # Wav2Vec2 outputs (batch, time, 768)
        
        # Projection to shared latent space
        self.projection = nn.Linear(self.backbone_feature_dim, embedding_dim)
        self.norm = nn.LayerNorm(embedding_dim)
        
        # Task-specific classifier (for pretraining)
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(embedding_dim, num_classes)
        )
        
        # Freeze backbone
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False
        
        self.embedding_dim = embedding_dim
    
    def forward(self, x: torch.Tensor):
        """
        x: (batch, samples) — raw waveform at 16kHz
        Returns: dict with 'embedding' and 'logits'
        """
        # Extract features: (batch, time, 768)
        outputs = self.backbone(x)
        features = outputs.last_hidden_state  # (batch, time, 768)
        
        # Pool over time dimension (mean pooling)
        # Same principle as adaptive_avg_pool2d in visual encoder
        features = features.mean(dim=1)  # (batch, 768)
        
        # Project to shared latent space
        embedding = self.projection(features)  # (batch, 512)
        embedding = self.norm(embedding)
        
        # Classifier (only used during pretraining)
        logits = self.classifier(embedding)
        
        return {
            "embedding": embedding,  # 512-dim for VATSA fusion
            "logits": logits        # for classification task
        }

# Initialize model (frozen backbone — Experiment 001)
model = VATSA_AudioEncoder(
    embedding_dim=512,
    backbone_name="wav2vec2_base",
    pretrained=True,
    freeze_backbone=True,  # FROZEN for baseline
    num_classes=num_classes
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")
print(f"Frozen params: {total_params - trainable_params:,}")

Loading weights: 100%|██████████| 210/210 [00:00<00:00, 8437.79it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total params: 94,784,419
Trainable params: 412,707
Frozen params: 94,371,712


## 5. Training Configuration

**Parallel to V-Module Exp 001:**
- Optimizer: AdamW, lr=1e-3
- Scheduler: CosineAnnealingLR
- Mixed precision: Yes (GradScaler)
- Epochs: 20
- Only projection + classifier trainable

In [15]:
# Initialize W&B (offline mode, same as V-Module)
wandb.init(
    project="vatsa-audio-2026",
    config={
        "experiment": "A-001_frozen_backbone",
        "backbone": "wav2vec2_base",
        "embedding_dim": 512,
        "learning_rate": 1e-3,
        "batch_size": batch_size,
        "epochs": 20,
        "optimizer": "AdamW",
        "scheduler": "CosineAnnealingLR",
        "freeze_backbone": True,
        "dataset": "speech_commands_v0.02",
        "num_classes": num_classes
    }
)

# Loss, optimizer, scheduler
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3,
    weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

# Mixed precision (same as V-Module)
scaler = torch.amp.GradScaler("cuda")

print("Training setup complete")

Training setup complete


In [ ]:
# Training loop (same structure as V-Module)
best_val_acc = 0.0
train_losses = []
val_accs = []

for epoch in range(20):
    model.train()
    epoch_loss = 0.0
    
    for batch_idx, (waveforms, labels) in enumerate(train_loader):
        waveforms = waveforms.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        # Mixed precision forward
        with torch.amp.autocast("cuda", dtype=torch.float16):
            outputs = model(waveforms)
            loss = criterion(outputs["logits"], labels)
        
        # Scaled backward
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        epoch_loss += loss.item()
        
        if batch_idx % 50 == 0:
            print(f"  Epoch {epoch+1} [{batch_idx}/{len(train_loader)}] Loss: {loss.item():.4f}", end="\r")
    
    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    
    # Validation
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for waveforms, labels in val_loader:
            waveforms = waveforms.to(device)
            labels = labels.to(device)
            
            with torch.amp.autocast("cuda", dtype=torch.float16):
                outputs = model(waveforms)
                _, predicted = torch.max(outputs["logits"], 1)
            
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    val_acc = 100 * correct / total
    val_accs.append(val_acc)
    
    # Step scheduler
    scheduler.step()
    
    # Log to W&B
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": avg_loss,
        "val_accuracy": val_acc,
        "learning_rate": optimizer.param_groups[0]["lr"]
    })
    
    print(f"Epoch {epoch+1}/20 — Loss: {avg_loss:.4f} — Val Acc: {val_acc:.2f}% — LR: {optimizer.param_groups[0]['lr']:.6f}")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        checkpoint = {
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "val_acc": val_acc,
            "config": {
                "embedding_dim": 512,
                "backbone": "wav2vec2_base",
                "num_classes": num_classes,
                "freeze_backbone": True
            }
        }
        torch.save(checkpoint, "vatsa_audio_encoder_speech_commands.pth")
        print(f"  ✓ Saved best model (val_acc: {val_acc:.2f}%)")

print(f"\nTraining complete. Best val accuracy: {best_val_acc:.2f}%")

## 6. Test Evaluation

**Parallel to V-Module:** Load best checkpoint, evaluate on test set.

In [16]:
# Load best checkpoint
checkpoint = torch.load("vatsa_audio_encoder_speech_commands.pth")

# Handle torch.compile prefix if present (same fix as V-Module)
state_dict = checkpoint["model_state"]
from collections import OrderedDict
new_state_dict = OrderedDict()
for k, v in state_dict.items():
    name = k.replace("_orig_mod.", "")
    new_state_dict[name] = v

model = VATSA_AudioEncoder(
    embedding_dim=512,
    backbone_name="wav2vec2_base",
    pretrained=True,
    freeze_backbone=True,
    num_classes=num_classes
).to(device)

model.load_state_dict(new_state_dict)
model.eval()

# Test evaluation
correct = 0
total = 0

with torch.no_grad():
    for waveforms, labels in test_loader:
        waveforms = waveforms.to(device)
        labels = labels.to(device)
        
        with torch.amp.autocast("cuda", dtype=torch.float16):
            outputs = model(waveforms)
            _, predicted = torch.max(outputs["logits"], 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_accuracy = 100 * correct / total
print(f"Test Accuracy: {test_accuracy:.2f}%")

# Log final result
wandb.log({"test_accuracy": test_accuracy})

FileNotFoundError: [Errno 2] No such file or directory: 'vatsa_audio_encoder_speech_commands.pth'

## 7. Extract 512-dim Embeddings for VATSA Fusion

**Parallel to V-Module:** Extract embeddings from train set for later cross-modal alignment.

In [ ]:
# Extract embeddings (same pattern as V-Module)
embeddings_list = []
labels_list = []

model.eval()
print("Generating embeddings...")

with torch.no_grad():
    for i, (waveforms, labels) in enumerate(train_loader):
        waveforms = waveforms.to(device)
        
        with torch.amp.autocast("cuda", dtype=torch.float16):
            outputs = model(waveforms)
            embeddings = outputs["embedding"]
        
        embeddings_list.append(embeddings.cpu())
        labels_list.append(labels)
        
        if (i + 1) % 10 == 0:
            print(f"  Batch {i+1}/{len(train_loader)} processed", end="\r")

# Concatenate and save
final_embeddings = torch.cat(embeddings_list)
final_labels = torch.cat(labels_list)

torch.save({
    "embeddings": final_embeddings,
    "labels": final_labels,
    "label_names": all_labels,
    "model_config": {
        "backbone": "wav2vec2_base",
        "embedding_dim": 512,
        "dataset": "speech_commands_v0.02"
    }
}, "speech_commands_embeddings.pt")

print(f"\nSuccess! Saved {final_embeddings.shape[0]} embeddings to speech_commands_embeddings.pt")
print(f"Embedding shape: {final_embeddings.shape}")  # Should be (N, 512)

# Log to W&B
artifact = wandb.Artifact("speech-commands-embeddings", type="embeddings")
artifact.add_file("speech_commands_embeddings.pt")
wandb.log_artifact(artifact)

## 8. Experiment Summary

**Parallel to V-Module Exp 001:**

In [ ]:
summary = {
    "experiment": "A-001_frozen_backbone_baseline",
    "date": datetime.now().isoformat(),
    "phase": "2 — A-Module",
    "backbone": "wav2vec2_base-960h",
    "dataset": "speech_commands_v0.02",
    "num_classes": num_classes,
    "embedding_dim": 512,
    "freeze_backbone": True,
    "trainable_params": trainable_params,
    "total_params": total_params,
    "best_val_accuracy": best_val_acc,
    "test_accuracy": test_accuracy,
    "epochs": 20,
    "batch_size": batch_size,
    "learning_rate": 1e-3,
    "optimizer": "AdamW",
    "scheduler": "CosineAnnealingLR",
    "mixed_precision": True,
    "checkpoints": [
        "vatsa_audio_encoder_speech_commands.pth",
        "speech_commands_embeddings.pt"
    ]
}

with open("a001_experiment_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("═══ VATSA A-Module — Experiment 001 Summary ═══")
print(f"Test Accuracy: {test_accuracy:.2f}%")
print(f"Best Val Accuracy: {best_val_acc:.2f}%")
print(f"Trainable Parameters: {trainable_params:,}")
print(f"Frozen Parameters: {total_params - trainable_params:,}")
print(f"Embedding Dimension: 512 (VATSA shared latent space)")
print("═══════════════════════════════════════════════")

# Finish W&B
wandb.finish()

## Next Step

→ **Experiment A-002:** Unfreeze last Transformer layers, reduce LR to 1e-4

**Expected outcome:** +10-15% accuracy improvement (mirroring V-Module's 79% → 94% jump)